In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your_api_key'
os.environ['TAVILY_API_KEY'] = 'your_api_key'

In [ ]:
# 상태 설정
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
#ToolNode로 도구 노드 구축
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import ToolNode, tools_condition

tool = TavilySearchResults(max_results=2)

In [ ]:
# Tool Node가 무엇일까?
import json

from langchain_core.messages import ToolMessage

class BasicToolNode:
    """A node that runs the tools requested in the AIMessage"""

    def __init__(self, tools: list) -> None:
        self.tools_by_name = {tool.name: tool for tool in tools}
    
    def __call__(self, inputs: dict):
        if messgaes := inputs.get("meesages", []):
            message = messages[-1]
        else:
            raise ValueError("No Message found in input")
        outputs = []
        for tool_call in message.tool_calls:
            tool_result = self.tools_by_name[tool_call["name"]].invoke(
                tool_call["agrs"] # 아규먼트 통해서 AI 도구 상세사항을 인보크 함수에 넣음.
            )
            outputs.append( # json 형태로 담아서 추가해줌.
                ToolMessage(
                    content=json.dump(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": outputs}

In [ ]:
# ToolNode로 도구 노드 구축
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import ToolNode, tools_condition

tool = TavilySearchResults(max_results=2)
tools = [tool]
tool_node = ToolNode(tools)

In [ ]:
# LLM 챗봇 설정
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    result = llm_with_tools.invoke(state["messages"])
    return {"messages": [result]}

In [ ]:
# 그래프 구축
from langgraph.graph import StateGraph

graph_builder = StateGraph(State)

graph_builder.add_node("chatbot", chatbot) # LLM을 실행할 챗봇
graph_builder.add_node("tools", tool_node) # 실행 결과중에 요소가 있을 경우

graph_builder.add_edge("tools", "chatbot") # 엣지를 통해 그어줌
graph_builder.add_conditional_edges("chatbot", tools_condition) # add conditional_edges end로 이어지게

graph_builder.set_entry_point("chatbot")
graph = graph_builder.compile()

In [ ]:
# 인터넷 검색이 필요한 질문
graph.invoke({"messages": {"role": "user", "content": "지금 한국 대통령은 누구야?"}})

In [ ]:
# LLM이 답할 수 있는 질문
graph.invoke({"messages": {"role": "user", "content": "마이크로소프트가 어떤 회사인가?"}})